<a href="https://colab.research.google.com/github/viraj97-sl/ai-ml-ds-learning-hub/blob/master/04_Foundations/mathematics/07_information_theory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 07 — Information Theory

Information theory quantifies uncertainty and information content. It directly powers the loss functions you use every day — cross-entropy loss is information theory. Decision trees use information gain (entropy reduction). GANs use KL divergence.

**Topics:**
- Shannon Entropy
- Joint entropy and conditional entropy
- Mutual information
- KL divergence
- Cross-entropy and why it's the right loss for classification
- Information gain in decision trees

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import entropy as scipy_entropy

np.random.seed(42)

## 1. Shannon Entropy

**H(X) = -Σ p(x) log₂ p(x)**

Entropy measures **average surprise** (uncertainty) in a distribution.
- High entropy = high uncertainty = more information needed
- Low entropy = predictable = less information needed
- Maximum entropy for n outcomes: log₂(n) bits (uniform distribution)

In [ ]:
def entropy(p, base=2):
    """Shannon entropy. p is a probability distribution (sums to 1)."""
    p = np.array(p)
    p = p[p > 0]  # ignore zero probabilities (0*log(0) = 0 by convention)
    return -np.sum(p * np.log(p) / np.log(base))

# Examples
print('Binary examples (coin flip):')
for p_heads in [0.0, 0.1, 0.25, 0.5, 0.75, 0.9, 1.0]:
    p = [p_heads, 1 - p_heads]
    h = entropy(p)
    print(f'  P(H)={p_heads:.2f}: H={h:.4f} bits', '← maximum!' if p_heads == 0.5 else '')

print('\nMulti-class examples:')
examples = {
    'Uniform (4 classes)': [0.25, 0.25, 0.25, 0.25],
    'Almost certain':      [0.97, 0.01, 0.01, 0.01],
    'Skewed':              [0.7,  0.1,  0.1,  0.1 ],
}
for name, dist in examples.items():
    print(f'  {name}: H={entropy(dist):.4f} bits (max={np.log2(len(dist)):.4f})')

# Plot entropy as function of p for binary case
p_range = np.linspace(0.001, 0.999, 200)
h_binary = [-p*np.log2(p) - (1-p)*np.log2(1-p) for p in p_range]

plt.figure(figsize=(8, 4))
plt.plot(p_range, h_binary, 'b-', lw=2)
plt.axvline(0.5, color='red', ls='--', label='Maximum entropy at p=0.5')
plt.xlabel('p (probability of heads)'); plt.ylabel('H(X) bits')
plt.title('Binary Entropy: H(p) = -p·log₂(p) - (1-p)·log₂(1-p)')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 2. KL Divergence — Measuring Distribution Difference

**KL(P || Q) = Σ p(x) log(p(x) / q(x))**

Measures the "extra bits" needed to encode P using Q.
- KL(P||Q) ≥ 0 always (Gibbs inequality)
- KL(P||Q) = 0 iff P = Q
- NOT symmetric: KL(P||Q) ≠ KL(Q||P)

In [ ]:
def kl_divergence(P, Q, base=np.e):
    """KL(P||Q) — P is true, Q is approximation."""
    P, Q = np.array(P), np.array(Q)
    # Add small epsilon for numerical stability
    eps = 1e-10
    P, Q = P + eps, Q + eps
    P, Q = P / P.sum(), Q / Q.sum()  # normalize
    return np.sum(P * np.log(P / Q) / np.log(base))

# Example: true distribution P, approximate with Q
P = np.array([0.4, 0.3, 0.2, 0.1])  # true
Q_good = np.array([0.38, 0.32, 0.18, 0.12])  # close approximation
Q_bad  = np.array([0.25, 0.25, 0.25, 0.25])  # uniform (wrong)
Q_worse = np.array([0.1, 0.1, 0.1, 0.7])     # very wrong

print('KL Divergences (P = true distribution):')
print(f'  KL(P||P)       = {kl_divergence(P, P):.6f} (zero — perfect)')
print(f'  KL(P||Q_good)  = {kl_divergence(P, Q_good):.6f} (close approximation)')
print(f'  KL(P||Q_bad)   = {kl_divergence(P, Q_bad):.6f} (uniform)')
print(f'  KL(P||Q_worse) = {kl_divergence(P, Q_worse):.6f} (very wrong)')

print('\nAsymmetry: KL(P||Q) ≠ KL(Q||P)')
print(f'  KL(P||Q_bad) = {kl_divergence(P, Q_bad):.4f}')
print(f'  KL(Q_bad||P) = {kl_divergence(Q_bad, P):.4f}')

# Visualize KL between two Gaussians
x = np.linspace(-5, 8, 500)
P_gauss = lambda x, mu, sigma: np.exp(-0.5*((x-mu)/sigma)**2) / (sigma*np.sqrt(2*np.pi))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

mu_shifts = [0, 0.5, 1, 2, 3]
kls_forward = []
kls_reverse = []
P_vals = P_gauss(x, 0, 1)

for shift in mu_shifts:
    Q_vals = P_gauss(x, shift, 1)
    dx = x[1] - x[0]
    # Numerical KL
    kl_fwd = np.sum(P_vals * np.log(P_vals / (Q_vals + 1e-10)) * dx)
    kl_rev = np.sum(Q_vals * np.log(Q_vals / (P_vals + 1e-10)) * dx)
    kls_forward.append(max(0, kl_fwd))
    kls_reverse.append(max(0, kl_rev))

ax = axes[0]
for shift, color in zip([0, 1, 3], ['steelblue', 'darkorange', 'green']):
    ax.plot(x, P_gauss(x, shift, 1), color=color, lw=2, label=f'N({shift}, 1)')
ax.set_title('Gaussian Distributions with Different Means')
ax.set_xlabel('x'); ax.legend(); ax.grid(True, alpha=0.3)

ax2 = axes[1]
ax2.plot(mu_shifts, kls_forward, 'b-o', lw=2, label='KL(P||Q) — forward')
ax2.plot(mu_shifts, kls_reverse, 'r-o', lw=2, label='KL(Q||P) — reverse')
ax2.set_xlabel('Mean shift of Q'); ax2.set_ylabel('KL Divergence (nats)')
ax2.set_title('KL Divergence as Gaussians Diverge')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

## 3. Cross-Entropy — The Classification Loss

**H(P, Q) = -Σ p(x) log q(x)**

**Relationship:** H(P, Q) = H(P) + KL(P || Q)

When P is the true labels (one-hot), H(P) = 0, so:
**Cross-entropy loss = KL divergence from true labels to predicted probabilities**

Minimizing cross-entropy loss IS minimizing the KL divergence!

In [ ]:
def cross_entropy(p_true, q_pred, base=np.e):
    """Cross-entropy H(P, Q)."""
    eps = 1e-10
    q_pred = np.clip(q_pred, eps, 1)
    return -np.sum(p_true * np.log(q_pred) / np.log(base))

# Binary classification example
print('Binary Cross-Entropy Loss')
print('True label: 1 (positive class)')
for p_pred in [0.99, 0.9, 0.7, 0.5, 0.3, 0.1, 0.01]:
    loss = cross_entropy([1, 0], [p_pred, 1-p_pred])
    print(f'  Predicted P(y=1)={p_pred:.2f}: loss={loss:.4f}', 
          ' ← nearly correct' if p_pred > 0.9 else (' ← WRONG!' if p_pred < 0.3 else ''))

# Visualize binary cross-entropy
p_pred_range = np.linspace(0.001, 0.999, 300)
bce_y1 = -np.log(p_pred_range)       # true label = 1
bce_y0 = -np.log(1 - p_pred_range)   # true label = 0

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(p_pred_range, bce_y1, 'b-', lw=2, label='Loss when true y=1')
ax.plot(p_pred_range, bce_y0, 'r-', lw=2, label='Loss when true y=0')
ax.set_ylim(0, 5); ax.set_xlabel('Predicted P(y=1)')
ax.set_ylabel('Cross-Entropy Loss'); ax.legend()
ax.set_title('Binary Cross-Entropy Loss\n(heavily penalizes confident wrong predictions)')
ax.grid(True, alpha=0.3)

# Softmax output for 3-class problem
ax2 = axes[1]
true_class = 0  # class 0 is correct
confidence = np.linspace(0.34, 0.99, 100)  # confidence in correct class
losses = []
for conf in confidence:
    remaining = (1 - conf) / 2  # split remaining prob equally
    pred = [conf, remaining, remaining]
    true = [1, 0, 0]
    losses.append(cross_entropy(true, pred))

ax2.plot(confidence, losses, 'g-', lw=2)
ax2.set_xlabel('Confidence in correct class')
ax2.set_ylabel('Cross-Entropy Loss')
ax2.set_title('3-Class Cross-Entropy: Loss vs. Correct Class Confidence')
ax2.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

## 4. Mutual Information

**I(X; Y) = H(X) + H(Y) - H(X, Y)**

Mutual information measures how much knowing X tells us about Y (and vice versa).
- I(X; Y) = 0 → X and Y are independent
- I(X; Y) = H(X) → knowing Y tells you everything about X

Used in: feature selection, decision tree splits (information gain), NLP

In [ ]:
# Information Gain in Decision Trees
# IG(feature) = H(parent) - weighted_avg(H(children))

def information_gain(parent_labels, split_groups):
    """Compute information gain of a split."""
    def h(labels):
        if len(labels) == 0: return 0
        _, counts = np.unique(labels, return_counts=True)
        probs = counts / len(labels)
        return -np.sum(probs * np.log2(probs + 1e-10))
    
    n_total = len(parent_labels)
    h_parent = h(parent_labels)
    weighted_h_children = sum(len(g) / n_total * h(g) for g in split_groups)
    return h_parent - weighted_h_children

# Example: predict if it rains (yes/no) based on humidity (high/low)
parent = np.array([1,1,1,1,1,0,0,0,0,0,1,1,0,0])  # rain labels

# Split 1: by humidity
high_humidity = np.array([1,1,1,1,0,0,1,1])  # high humidity days
low_humidity  = np.array([1,0,0,0,0,0])       # low humidity days
ig_humidity = information_gain(parent, [high_humidity, low_humidity])

# Split 2: random (useless split)
group_a = parent[:7]
group_b = parent[7:]
ig_random = information_gain(parent, [group_a, group_b])

_, counts = np.unique(parent, return_counts=True)
probs = counts / len(parent)
h_parent = -np.sum(probs * np.log2(probs + 1e-10))

print('Decision Tree Information Gain Example')
print(f'Parent entropy H(Rain): {h_parent:.4f} bits')
print(f'IG(Humidity split):     {ig_humidity:.4f} bits')
print(f'IG(Random split):       {ig_random:.4f} bits')
print(f'Best feature: Humidity (higher IG = more informative split)')

# Visualize relationship between features and MI
np.random.seed(42)
n_pts = 300
correlations = [0, 0.3, 0.7, 0.99]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for idx, rho in enumerate(correlations):
    cov = [[1, rho], [rho, 1]]
    data = np.random.multivariate_normal([0, 0], cov, n_pts)
    x_d, y_d = data[:, 0], data[:, 1]
    # Approximate MI via correlation for Gaussian: MI = -0.5 * log(1 - rho^2)
    mi_approx = -0.5 * np.log(1 - rho**2) if abs(rho) < 1 else float('inf')
    axes[idx].scatter(x_d, y_d, alpha=0.3, s=10)
    axes[idx].set_title(f'ρ={rho}\nMI≈{mi_approx:.3f} nats')
    axes[idx].set_xlabel('X'); axes[idx].set_ylabel('Y')
    axes[idx].grid(True, alpha=0.3)

plt.suptitle('Mutual Information Increases with Correlation', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

## Summary

| Concept | Formula | ML Application |
|---------|---------|---------------|
| Shannon Entropy | H(X) = -Σ p·log p | Decision tree splits, measuring uncertainty |
| KL Divergence | KL(P\|\|Q) = Σ p·log(p/q) | VAE loss, model comparison, distributional RL |
| Cross-Entropy | H(P,Q) = H(P) + KL(P\|\|Q) | **Classification loss function** |
| Mutual Information | I(X;Y) = H(X)+H(Y)-H(X,Y) | Feature selection, information bottleneck |
| Information Gain | H(parent) - H(children) | Decision tree, Random Forest feature splits |

**Key insight:** Cross-entropy loss = KL divergence from predictions to true labels. Minimizing your neural network's cross-entropy loss is mathematically equivalent to making your model's distribution as close as possible to the true data distribution.

---

**You've completed the Mathematics Foundation track!** 🎉

Next steps:
- [Data Scientist Track →](../../01_Data_Scientist/README.md)
- [ML Engineer Track →](../../02_ML_Engineer/README.md)